# 🤖 Classification Algorithms & Model Tuning

This notebook covers:
- **Logistic Regression** — probabilistic binary classifier
- **K-Nearest Neighbors (KNN)** — instance-based learning
- **Decision Tree** — rule-based tree structure
- **Random Forest** — ensemble of decision trees
- **Support Vector Machine (SVM)** — maximum margin classifier
- **Model Evaluation** — confusion matrix, classification report, ROC-AUC
- **Hyperparameter Tuning** — GridSearchCV and RandomizedSearchCV
- **Model Comparison** dashboard

> 🎯 **Goal:** Learn multiple classification approaches, evaluate them properly, and tune them for peak performance.

---
## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Datasets
from sklearn.datasets import load_breast_cancer, make_classification

# Preprocessing
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

print('✅ All libraries imported successfully!')

**Expected Output:**
```
✅ All libraries imported successfully!
```

---
## 📊 2. Load & Explore the Dataset

We use the **Breast Cancer Wisconsin** dataset:
- 569 samples, 30 features (cell nucleus measurements)
- Binary target: **0 = Malignant**, **1 = Benign**
- Classic benchmark for binary classification

In [ ]:
# --------------------------------------------------
# Load Breast Cancer dataset and build a DataFrame
# --------------------------------------------------

cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print('📋 Dataset Info:')
print(f'   Shape        : {df.shape}')
print(f'   Features     : {len(cancer.feature_names)}')
print(f'   Classes      : {list(cancer.target_names)}')
print(f'   Missing values: {df.isnull().sum().sum()}\n')

print('📊 Class Distribution:')
print(df['diagnosis'].value_counts().to_string())
print(f'\n   Malignant: {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')
print(f'   Benign   : {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')

**Expected Output:**
```
📋 Dataset Info:
   Shape        : (569, 32)
   Features     : 30
   Classes      : ['malignant' 'benign']
   Missing values: 0

📊 Class Distribution:
   Benign      357
   Malignant   212
   Malignant: 212 (37.3%)
   Benign   : 357 (62.7%)
```
Slight class imbalance — more benign cases than malignant.

In [ ]:
# --------------------------------------------------
# Visualize class distribution and key features
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Class distribution bar chart ---
class_counts = df['diagnosis'].value_counts()
axes[0].bar(class_counts.index, class_counts.values,
            color=['salmon', 'steelblue'], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count', fontsize=12)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# --- Feature boxplot: 'mean radius' by diagnosis ---
df.boxplot(column='mean radius', by='diagnosis', ax=axes[1],
           patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Mean Radius by Diagnosis', fontsize=13)
axes[1].set_xlabel('Diagnosis', fontsize=12)
axes[1].set_ylabel('Mean Radius', fontsize=12)

plt.suptitle('')   # Remove default suptitle from boxplot
plt.tight_layout()
plt.show()

**Expected Output:**
- **Left:** Bar chart showing ~357 Benign vs ~212 Malignant cases
- **Right:** Malignant tumors have larger mean radius than benign ones — great discriminating feature!

In [ ]:
# --------------------------------------------------
# Prepare data: split & scale
# --------------------------------------------------

X = df[cancer.feature_names]
y = df['target']

# Stratified split preserves class ratio in train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standard scaling — zero mean, unit variance
# Essential for distance-based and gradient-based algorithms (KNN, SVM, LR)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train class ratio: Malignant={sum(y_train==0)}, Benign={sum(y_train==1)}')
print(f'Test  class ratio: Malignant={sum(y_test==0)},  Benign={sum(y_test==1)}')

**Expected Output:**
```
Train: (455, 30) | Test: (114, 30)
Train class ratio: Malignant=169, Benign=286
Test  class ratio: Malignant=43,  Benign=71
```
Stratification ensures both sets maintain ~37/63 class balance.

---
## 🔵 3. Logistic Regression

Logistic Regression applies the **sigmoid function** to output probabilities:
$$P(y=1|X) = \sigma(X\theta) = \frac{1}{1+e^{-X\theta}}$$

Decision boundary: predict class 1 if $P > 0.5$, else class 0.

In [ ]:
# --------------------------------------------------
# Train Logistic Regression
# C = inverse of regularization strength (larger C = less regularization)
# --------------------------------------------------

log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)
y_pred_lr = log_reg.predict(X_test_sc)

print('🔵 Logistic Regression Results:')
print(f'   Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Malignant', 'Benign']))

**Expected Output:**
```
🔵 Logistic Regression Results:
   Accuracy: ~97-98%

Classification Report:
              precision  recall  f1-score  support
   Malignant     0.97      0.95     0.96       43
      Benign     0.97      0.99     0.98       71
    accuracy                        0.97      114
```
Logistic Regression performs very well on this linearly separable dataset.

---
## 🟢 4. K-Nearest Neighbors (KNN)

KNN classifies a sample by looking at its **K nearest neighbors** in feature space.
- Non-parametric — no explicit model is learned
- Sensitive to feature scale → always scale before using KNN

In [ ]:
# --------------------------------------------------
# Find optimal K using cross-validation
# --------------------------------------------------

k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    # 5-fold cross-validation on training set
    scores = cross_val_score(knn, X_train_sc, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]

# Plot K vs accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_range, cv_scores, 'bo-', linewidth=2, markersize=6)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('KNN: Choosing Optimal K via Cross-Validation', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Train with best K
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred_knn = knn_best.predict(X_test_sc)

print(f'🟢 KNN (K={best_k}) Accuracy: {accuracy_score(y_test, y_pred_knn)*100:.2f}%')

**Expected Output:**
- A curve showing accuracy peaks around K=5–15
```
🟢 KNN (K=xx) Accuracy: ~95-97%
```

---
## 🟡 5. Decision Tree

A Decision Tree splits data at each node based on the feature that maximizes **information gain** (or minimizes Gini impurity).

Key hyperparameters:
- `max_depth` — limits tree depth to prevent overfitting
- `min_samples_split` — minimum samples to split a node
- `criterion` — 'gini' or 'entropy'

In [ ]:
# --------------------------------------------------
# Train Decision Tree & visualize first 3 levels
# --------------------------------------------------

dt = DecisionTreeClassifier(max_depth=4, random_state=42, criterion='gini')
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)

print(f'🟡 Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt)*100:.2f}%')
print(f'   Tree depth: {dt.get_depth()} | Leaf nodes: {dt.get_n_leaves()}')

# Visualize the tree (first 3 levels for readability)
plt.figure(figsize=(20, 8))
plot_tree(
    dt, max_depth=3,
    feature_names=cancer.feature_names,
    class_names=['Malignant', 'Benign'],
    filled=True, rounded=True, fontsize=9
)
plt.title('Decision Tree (first 3 levels)', fontsize=14)
plt.tight_layout()
plt.show()

**Expected Output:**
```
🟡 Decision Tree Accuracy: ~92-95%
   Tree depth: 4 | Leaf nodes: xx
```
A colorful tree diagram showing splitting conditions (e.g., 'worst radius <= 16.7') — blue nodes = Benign, orange = Malignant.

In [ ]:
# --------------------------------------------------
# Overfitting demonstration: max_depth vs accuracy
# Deep trees memorize training data (overfit)
# --------------------------------------------------

depths = range(1, 20)
train_accs, test_accs = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_sc, y_train)
    train_accs.append(accuracy_score(y_train, tree.predict(X_train_sc)))
    test_accs.append(accuracy_score(y_test, tree.predict(X_test_sc)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'bo-', label='Train Accuracy', linewidth=2)
plt.plot(depths, test_accs, 'rs-', label='Test Accuracy', linewidth=2)
plt.xlabel('Max Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree: Overfitting with Increasing Depth', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('📌 Insight: Train accuracy reaches 100% at high depths → Overfitting!')
print('   Test accuracy peaks at a moderate depth (sweet spot)')

**Expected Output:**
- Train accuracy climbs to 100% at large depths — clear **overfitting**
- Test accuracy peaks around depth 3–6, then plateaus or drops

---
## 🌲 6. Random Forest

Random Forest is an **ensemble** of decision trees:
- Each tree is trained on a **bootstrapped** sample
- Each split considers a **random subset of features**
- Final prediction = **majority vote** of all trees

This reduces variance (overfitting) while keeping low bias.

In [ ]:
# --------------------------------------------------
# Train Random Forest
# n_estimators = number of trees in the forest
# --------------------------------------------------

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

print(f'🌲 Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%')

# Feature importance: how much each feature reduces impurity across all trees
feat_imp = pd.Series(rf.feature_importances_, index=cancer.feature_names)
top_features = feat_imp.nlargest(10)

plt.figure(figsize=(9, 5))
top_features.sort_values().plot(kind='barh', color='forestgreen', edgecolor='white')
plt.xlabel('Feature Importance (Mean Impurity Decrease)', fontsize=11)
plt.title('Random Forest — Top 10 Feature Importances', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(top_features.head().to_string())

**Expected Output:**
```
🌲 Random Forest Accuracy: ~96-98%
```
- Feature importance chart showing **worst concave points**, **worst radius**, and **worst perimeter** as top predictors
- Random Forest typically outperforms a single Decision Tree

---
## 🔴 7. Support Vector Machine (SVM)

SVM finds the **hyperplane** that maximizes the margin between classes.
- Uses **kernel trick** to handle non-linear boundaries (RBF, polynomial)
- `C` = regularization (trade-off between margin width and misclassification)
- `gamma` = kernel coefficient (how far influence of a single training sample reaches)

In [ ]:
# --------------------------------------------------
# Train SVM with RBF kernel
# probability=True enables predict_proba for ROC curves
# --------------------------------------------------

svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_sc, y_train)
y_pred_svm = svm.predict(X_test_sc)

print(f'🔴 SVM (RBF kernel) Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_svm, target_names=['Malignant', 'Benign']))

**Expected Output:**
```
🔴 SVM (RBF kernel) Accuracy: ~97-99%
```
SVM with RBF kernel is one of the top performers on this dataset.

---
## 📊 8. Model Evaluation — Confusion Matrix & ROC Curves

### Confusion Matrix tells us:
- **TP** — correctly predicted positive (benign → benign)
- **FP** — predicted positive, actually negative (false alarm)
- **FN** — predicted negative, actually positive (**miss** — dangerous in medicine!)
- **TN** — correctly predicted negative

In [ ]:
# --------------------------------------------------
# Plot confusion matrices for all classifiers
# --------------------------------------------------

models_preds = {
    'Logistic\nRegression': y_pred_lr,
    'KNN': y_pred_knn,
    'Decision\nTree': y_pred_dt,
    'Random\nForest': y_pred_rf,
    'SVM': y_pred_svm
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, pred) in zip(axes, models_preds.items()):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Malignant', 'Benign'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, pred)
    ax.set_title(f'{name}\nAcc: {acc*100:.1f}%', fontsize=10)

plt.suptitle('Confusion Matrices — All Classifiers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Expected Output:**
5 confusion matrices side by side. Look for:
- **False Negatives** (bottom-left cell) — missed malignant tumors are the most critical errors in cancer detection
- SVM and Random Forest should have the fewest off-diagonal entries

In [ ]:
# --------------------------------------------------
# ROC Curves — visualize discrimination ability
# AUC = Area Under Curve (1.0 = perfect, 0.5 = random)
# --------------------------------------------------

models_prob = {
    'Logistic Regression': log_reg,
    'KNN':                 knn_best,
    'Decision Tree':       dt,
    'Random Forest':       rf,
    'SVM':                 svm
}

plt.figure(figsize=(9, 7))

for name, model in models_prob.items():
    y_proba = model.predict_proba(X_test_sc)[:, 1]   # Probability of class 1 (Benign)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

# Diagonal = random classifier baseline
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC=0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curves — All Classifiers', fontsize=13)
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

**Expected Output:**
- All model curves should hug the top-left corner (high AUC ~0.98–1.00)
- **Random Forest** and **SVM** typically have the highest AUC scores
- **Decision Tree** may have slightly lower AUC due to instability

---
## ⚙️ 9. Hyperparameter Tuning

### Strategy Comparison:

| Method | How it works | Pros | Cons |
|---|---|---|---|
| **GridSearchCV** | Exhaustively tries all combinations | Thorough | Expensive for large grids |
| **RandomizedSearchCV** | Randomly samples combinations | Faster, often similar results | May miss best combo |
| **Bayesian Optimization** | Builds probabilistic model | Most efficient | More complex to set up |

In [ ]:
# --------------------------------------------------
# GridSearchCV on SVM
# Tests every combination of C × gamma × kernel
# Uses 5-fold cross-validation for each combo
# --------------------------------------------------

param_grid_svm = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf', 'linear']
}

grid_search_svm = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search_svm.fit(X_train_sc, y_train)

print('⚙️  GridSearchCV (SVM) Results:')
print(f'   Best parameters : {grid_search_svm.best_params_}')
print(f'   Best CV accuracy: {grid_search_svm.best_score_*100:.2f}%')

# Evaluate tuned model on test set
y_pred_svm_tuned = grid_search_svm.predict(X_test_sc)
print(f'   Test accuracy   : {accuracy_score(y_test, y_pred_svm_tuned)*100:.2f}%')

**Expected Output:**
```
⚙️  GridSearchCV (SVM) Results:
   Best parameters : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
   Best CV accuracy: ~98.xx%
   Test accuracy   : ~98-99%
```
GridSearchCV often finds C=10 as optimal, providing a small accuracy boost over the default.

In [ ]:
# --------------------------------------------------
# RandomizedSearchCV on Random Forest
# More efficient than grid search for large search spaces
# n_iter controls how many combinations to try
# --------------------------------------------------

param_dist_rf = {
    'n_estimators':      [50, 100, 200, 300],
    'max_depth':         [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None]
}

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=30,          # Try 30 random combinations
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search_rf.fit(X_train_sc, y_train)

print('🎲 RandomizedSearchCV (Random Forest) Results:')
print(f'   Best parameters : {random_search_rf.best_params_}')
print(f'   Best CV accuracy: {random_search_rf.best_score_*100:.2f}%')

y_pred_rf_tuned = random_search_rf.predict(X_test_sc)
print(f'   Test accuracy   : {accuracy_score(y_test, y_pred_rf_tuned)*100:.2f}%')

**Expected Output:**
```
🎲 RandomizedSearchCV (Random Forest) Results:
   Best parameters : {'n_estimators': 200, 'max_depth': None, ...}
   Best CV accuracy: ~97-98%
   Test accuracy   : ~97-99%
```

In [ ]:
# --------------------------------------------------
# Visualize GridSearch Results: C vs Accuracy
# --------------------------------------------------

cv_results = pd.DataFrame(grid_search_svm.cv_results_)

# Filter only RBF kernel results for cleaner plot
rbf_results = cv_results[cv_results['param_kernel'] == 'rbf'].copy()
rbf_results['param_C'] = rbf_results['param_C'].astype(float)

plt.figure(figsize=(9, 5))
for gamma in rbf_results['param_gamma'].unique():
    subset = rbf_results[rbf_results['param_gamma'] == gamma].sort_values('param_C')
    plt.plot(subset['param_C'], subset['mean_test_score'],
             marker='o', linewidth=2, label=f'gamma={gamma}')

plt.xscale('log')
plt.xlabel('C (Regularization)', fontsize=12)
plt.ylabel('Mean CV Accuracy', fontsize=12)
plt.title('SVM GridSearch: C vs Accuracy by Gamma (RBF kernel)', fontsize=13)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Expected Output:**
- Multiple lines showing accuracy across C values for each gamma setting
- Accuracy generally increases with C, then plateaus — shows why C=10 is often optimal

---
## 🏆 10. Model Comparison Dashboard

In [ ]:
# --------------------------------------------------
# Collect metrics for all models (default + tuned)
# --------------------------------------------------

from sklearn.metrics import precision_score, recall_score, f1_score

all_models = {
    'Logistic Regression':      y_pred_lr,
    'KNN':                      y_pred_knn,
    'Decision Tree':             y_pred_dt,
    'Random Forest':             y_pred_rf,
    'SVM (Default)':             y_pred_svm,
    'SVM (Tuned)':               y_pred_svm_tuned,
    'Random Forest (Tuned)':     y_pred_rf_tuned,
}

results = []
for name, pred in all_models.items():
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred) * 100, 2),
        'Precision': round(precision_score(y_test, pred) * 100, 2),
        'Recall':    round(recall_score(y_test, pred) * 100, 2),
        'F1 Score':  round(f1_score(y_test, pred) * 100, 2)
    })

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)

print('🏆 Model Comparison Dashboard:')
display(results_df.set_index('Model').style
        .background_gradient(cmap='YlGn', axis=0)
        .format('{:.2f}%'))

**Expected Output:**
A color-coded table like:
```
Model                    Accuracy  Precision  Recall   F1 Score
SVM (Tuned)              98.25%    98.59%     98.59%   98.59%
Random Forest (Tuned)    97.37%    97.18%     98.59%   97.88%
SVM (Default)            97.37%    ...         ...      ...
...
Decision Tree            93.xx%    ...         ...      ...
```
Tuned models consistently outperform their defaults.

In [ ]:
# --------------------------------------------------
# Grouped bar chart comparing models
# --------------------------------------------------

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(results_df))
width = 0.2

fig, ax = plt.subplots(figsize=(15, 6))

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Comparison — All Classifiers', fontsize=14)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
ax.set_ylim([88, 101])
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Expected Output:**
A grouped bar chart showing all 4 metrics side-by-side for each model. Tuned SVM and Random Forest bars should be the tallest.

In [ ]:
# --------------------------------------------------
# Cross-Validation comparison — unbiased estimate
# 10-fold CV gives a better estimate of true generalization
# --------------------------------------------------

cv_models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=best_k),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM':                 SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
}

cv_results_list = []
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=10, scoring='accuracy', n_jobs=-1)
    cv_results_list.append({'Model': name, 'Mean CV Accuracy': scores.mean(), 'Std': scores.std()})
    print(f'{name:25s}: {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%')

# Plot with error bars
cv_df = pd.DataFrame(cv_results_list).sort_values('Mean CV Accuracy', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(cv_df['Model'], cv_df['Mean CV Accuracy']*100,
         xerr=cv_df['Std']*100, color='steelblue', capsize=5, edgecolor='white')
plt.xlabel('10-Fold CV Accuracy (%)', fontsize=12)
plt.title('Cross-Validation Comparison (10-Fold)', fontsize=13)
plt.xlim([88, 101])
plt.tight_layout()
plt.show()

**Expected Output:**
```
Logistic Regression      : 97.xx% ± 1.xx%
KNN                      : 96.xx% ± 1.xx%
Decision Tree            : 92.xx% ± 2.xx%
Random Forest            : 96.xx% ± 1.xx%
SVM                      : 98.xx% ± 1.xx%
```
- A horizontal bar chart with error bars showing variance across folds
- Smaller error bar = more **stable** model across different data splits

---
## ✅ 11. Summary & Key Takeaways

| Algorithm | Best For | Key Strength | Watch Out For |
|---|---|---|---|
| **Logistic Regression** | Linearly separable data, interpretability | Fast, probabilistic output | Poor on complex boundaries |
| **KNN** | Small datasets, non-linear boundaries | Simple, no training phase | Slow at inference, needs scaling |
| **Decision Tree** | Interpretability, mixed data | Visual, handles non-linearity | High variance, overfits easily |
| **Random Forest** | General purpose | Robust, low variance, feature importance | Slower, less interpretable |
| **SVM** | High-dimensional data, small datasets | Excellent margin maximization | Slow on large data, needs scaling |

### Hyperparameter Tuning Key Points:
- Always tune on **cross-validation**, never on the test set
- **GridSearchCV** = exhaustive, best for small grids
- **RandomizedSearchCV** = faster, good for large search spaces
- Tuning typically improves accuracy by **1–3%** on well-designed baselines

---
*🎓 End of Classification Algorithms & Model Tuning Notebook*